# Notebook consacré à l'analyse des variables textuelles ouvertes

Exemple :




In [ ]:

!python -m spacy download fr_core_news_md

In [ ]:
import pandas as pd
import numpy as np
import spacy
import re

import nltk
from nltk.corpus import stopwords

import os #pour naviguer dans les dossiers
from io import StringIO
import s3fs #pour connecter au bucket

In [ ]:


# Create filesystem object
S3_ENDPOINT_URL = "https://" + os.environ["AWS_S3_ENDPOINT"]
fs = s3fs.S3FileSystem(client_kwargs={'endpoint_url': S3_ENDPOINT_URL})

BUCKET_OUT = "aluneau"



In [ ]:
with fs.open(f"{BUCKET_OUT}/data_eP8/raw/anonymous_answer_2026-04-20.csv", "r") as file_in:
    df0 = pd.read_csv(file_in, sep =",")


df_col = pd.read_csv("../le_questionnaire/dico_variable.csv", sep = ",")
df0.columns

In [ ]:
df_col["name"] = df_col.name.apply(lambda row : row.replace(' :', ''))

In [ ]:

list_text = [x for x in df_col.label.loc[(df_col.type=="texte_libre")&(df_col.label!="q12_1_how_humanum")&(df_col.personal_data==False)]]



dict_col = {}
for col in list_text:
    dict_col[col] = len(df0.loc[~df0[col].isna()])
print(dict_col)

In [ ]:
nb_rep_question_ouverte = pd.DataFrame.from_dict(data = dict_col, orient='index')
nb_rep_question_ouverte

In [ ]:
nlp = spacy.load('fr_core_news_md')
nlp.add_pipe("merge_entities")
#nlp.add_pipe("merge_noun_chunks")
spacy_stopwords = list(nlp.Defaults.stop_words)




nltk.download('stopwords')

spacy_stopwords
stop_words = set(stopwords.words('french'))


In [ ]:

for n, x in enumerate(df0.q9_utilite_accomp.loc[~df0.q9_utilite_accomp.isna()]):
    print(n+1, x)

# Traitement texte long

In [ ]:
df_test = df0[["q6_1_dev_exemple"]].loc[~df0["q6_1_dev_exemple"].isna()].copy()

    

In [ ]:
def tokenizer(data, column, list_stopword = None, pos=['NOUN','VERB','ADJ','PROPN']):
    dict_token = {}
    df_test = data[[column]].loc[~data[column].isna()].copy()
    for row in df_test[column]:
        new_row = re.sub("·",".", row)
        doc = nlp(new_row)
        if len(doc) > 2:
            cleaned_doc = "|".join([t.text.lower() for t in doc if t.pos_ in pos])
        
        else:
            cleaned_doc = " ".join([t.text.lower() for t in doc])
    #chunk_l = [chunk.text.lower() for chunk in doc.noun_chunks]
        dict_token[row] = cleaned_doc
    return dict_token

In [ ]:
tokenizer(df0, 'q37_raison_util_ia', list_stopword = None, pos=['NOUN','VERB','ADJ','PROPN'])

In [ ]:
df_clean = tokenizer(df0, column = "q9_utilite_accomp", list_stopword = stop_words)
df_clean[["cleaned_text", "tokens"]]

## traitement texte court

In [ ]:
df0["q5_1_other_lang_clean"] = df0.apply(lambda row: re.sub(r",", "", str(row.q5_1_other_lang)).lower(), 1) # remove urls
df0["q5_1_other_lang_clean"] = df0.apply(lambda row: re.sub(r"\bet\b", "", str(row.q5_1_other_lang_clean)).lower(), 1) # remove urls
df0["q5_1_other_lang_clean"] = df0.apply(lambda row: "".join(str(row.q5_1_other_lang_clean).split(". ")[0]), 1) # remove urls
df0["tokens"] = df0.apply(lambda row: row.q5_1_other_lang_clean.split(),1) 
df0[["q5_1_other_lang_clean","tokens"]]

In [ ]:
for n, x in enumerate(df0.q5_1_other_lang_clean.loc[~df0.q5_1_other_lang.isna()]):
    print(n+1, x)

# Types de données

In [ ]:
split_multiple_choices(df0, column='q33_sauvegarde_data', index = "q45_clé")

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

In [ ]:
# Initialize the matplotlib figure
fig, ax = plt.subplots(8, figsize=(10, 15))

# Plot the total crashes
sns.set_color_codes("pastel")

for n, col in enumerate(list_nominal_multiple):
    gb_data = split_multiple_choices(df0, column=col, index = "q45_clé")
    sns.barplot(x="total", y=col, data=gb_data,
                label="Total", color="b", ax=ax[n])
    sns.barplot(x="nb", y=col, data=gb_data,
                label="effectif", color="r", ax=ax[n])
    

sns.despine(left=True, bottom=True)
plt.savefig("multiple_choice.png")

In [ ]:
sns.barplot(x="nb", y=col, data=gb_data,
                label="effectif", color="r", ax=ax[n])